# GitHub ↔ Google Colab

Este notebook conecta o **[vision-transfer-learning](https://github.com/Gengo250/vision-transfer-learning)** ao ambiente do Colab.

1. Execute a próxima célula: na primeira vez, ela clona a branch `main`; nas seguintes, executa `git pull --ff-only`.
2. Os arquivos ficam em `/content/vision-transfer-learning` durante esta sessão.
3. Execute a célula final para conferir a branch, o commit e os arquivos.

> **Importante:** abrir este notebook pelo GitHub não sincroniza arquivos automaticamente. Para salvar alterações **deste notebook** de volta no repositório, use **Arquivo → Salvar uma cópia no GitHub**, selecionando `Gengo250/vision-transfer-learning` e `main`. Isso cria um commit no GitHub. Não inclui automaticamente alterações em outros arquivos do clone.

> O Colab usa armazenamento temporário. Ao reiniciar ou encerrar o ambiente de execução, execute a célula de sincronização novamente. Não salve datasets, checkpoints ou resultados importantes apenas em `/content`.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Gengo250/vision-transfer-learning.git"
BRANCH = "main"
REPO_DIR = Path("/content/vision-transfer-learning")

def git(*args, cwd=None):
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} existe, mas nao e um repositorio Git.")

    current_branch = git("branch", "--show-current", cwd=REPO_DIR)
    if current_branch != BRANCH:
        raise RuntimeError(
            f"Branch atual: {current_branch}. Esperada: {BRANCH}. "
            "Troque de branch manualmente antes de atualizar."
        )

    changes = git("status", "--porcelain", cwd=REPO_DIR)
    if changes:
        raise RuntimeError(
            "Ha alteracoes locais no clone do Colab. "
            "Salve-as antes de executar git pull para evitar perda de trabalho.\n"
            + changes
        )

    print("Atualizando o repositorio...")
    print(git("pull", "--ff-only", "origin", BRANCH, cwd=REPO_DIR))
else:
    print("Clonando o repositorio...")
    print(git("clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)))

os.chdir(REPO_DIR)
print(f"\nDiretorio: {Path.cwd()}")
print("Commit:", git("log", "-1", "--format=%h %s", cwd=REPO_DIR))


## Verificar sincronização

Esta célula não altera o repositório. Ela apenas mostra se o clone está na branch esperada e quais arquivos foram baixados.


In [ ]:
print("Branch:", git("branch", "--show-current", cwd=REPO_DIR))
print("Status:")
print(git("status", "--short", cwd=REPO_DIR) or "(sem alteracoes locais)")
print("\nArquivos na raiz:")
for path in sorted(REPO_DIR.iterdir()):
    if path.name != ".git":
        print("📁" if path.is_dir() else "📄", path.name)
